# Stage A3 — Normalization & Train/Validation/Test Split

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR
Supports dissertation Section 3.1.3.2 (feedback item asking to justify
the 70-15-15 split), Section 3.1.2.3 (Eq. 3.1, normalization).

Self-contained: reads `masters_data.xlsx` directly, no dependency on
A1/A2 outputs, no thesis-text reference values (this stage isn't a
text-vs-data check, so there's nothing here to compare against).

**Input:** `data/masters_data.xlsx`
**Output:** `split` label (train/val/test) and min-max normalized
columns per row; nothing written to disk unless you run the optional
save cell at the end.

**Split design in one sentence:** every row gets an *extremity score*
(how close it sits to the min or max of whichever input was swept for
it, Sec. 3.1.1.2's OFAT design), and the 6 highest-extremity rows
become the test set, the next 6 become validation, and the rest — the
interior/bulk of the design — become train. That directly targets the
"stratified split preserving extremes in test" requirement instead of
a plain random 70/15/15.


## Setup

In [ ]:
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

print("polars ", pl.__version__)
import plotly
print("plotly ", plotly.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
OUT_DIR = PROJECT_ROOT / "outputs"

SEED = 42  # fixes tie-breaking below so the split is reproducible run to run
RAW_PATH


## 1. Load data

In [ ]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI", "Lambda [-]": "lambda", "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail", "HC [g/kW.h]": "HC", "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2", "SO_H [FSN]": "PM", "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS

df = pl.read_excel(RAW_PATH).rename(COLUMN_MAP).select(ALL_COLS)
n = df.shape[0]
print(df.shape)
df.head()


## 2. Recompute OFAT block per row

Same data-driven method as A1 (deviation from each input's own median,
normalised by its range, with neighbour-smoothing) — repeated here
rather than imported, so this notebook has no dependency on A1 having
been run first.

In [ ]:
medians = {c: df[c].median() for c in INPUT_COLS}
ranges = {c: (df[c].max() - df[c].min()) for c in INPUT_COLS}

deviation = np.column_stack([
    np.abs(df[c].to_numpy() - medians[c]) / ranges[c] for c in INPUT_COLS
])
raw_block = np.array(INPUT_COLS)[deviation.argmax(axis=1)]


def smooth_isolated_labels(labels, passes=2):
    out = list(labels)
    for _ in range(passes):
        changed = False
        for i in range(1, len(out) - 1):
            if out[i] != out[i - 1] and out[i - 1] == out[i + 1]:
                out[i] = out[i - 1]
                changed = True
        if not changed:
            break
    return np.array(out)


ofat_block = smooth_isolated_labels(raw_block)
df = df.with_columns(pl.Series("ofat_block", ofat_block))
print(np.unique(ofat_block, return_counts=True))


**Equation implemented above** — for input $c$ and row $i$:

$$
\text{deviation}_{i,c} = \frac{\left| x_{i,c} - \text{median}(x_{:,c}) \right|}{\max(x_{:,c}) - \min(x_{:,c})}
\qquad\Longrightarrow\qquad
\text{block}_i = \arg\max_{c} \; \text{deviation}_{i,c}
$$

i.e. whichever input sits furthest from its own dataset median, scaled
by its own range, is taken as the swept variable for that row.

In [ ]:
fig = go.Figure(go.Bar(
    x=list(dict(zip(*np.unique(ofat_block, return_counts=True))).keys()),
    y=list(dict(zip(*np.unique(ofat_block, return_counts=True))).values()),
    marker_color=["#185FA5", "#993C1D", "#534AB7", "#3B6D11"][:len(np.unique(ofat_block))],
))
fig.update_layout(title="Rows per inferred OFAT block", yaxis_title="count",
                   width=550, height=350)
fig.show()


## 3. Extremity score & split assignment

For each row, rank it within its own OFAT block by the value of that
block's swept variable, and convert the rank to a 0-1 position (0 = the
block's minimum, 1 = its maximum):

$$
\text{position}_i = \frac{\text{rank}_i}{m_b - 1}, \qquad
\text{extremity}_i = \left| \text{position}_i - 0.5 \right| \times 2
$$

where $m_b$ is the size of row $i$'s block — so a block's extremes
score 1.0 and its central point scores 0.0.

Sort **all 40 rows together** by extremity, descending. The top 6
become test, the next 6 become validation, the remaining 28 become
train — 70/15/15 by construction, and test is deliberately weighted
toward domain boundaries rather than a plain random draw.

Ties at extremity = 1.0 (several blocks' min/max compete for the same
few slots) are broken by a seeded random jitter, so the exact rows
selected are reproducible but not hand-picked.

In [ ]:
extremity = np.zeros(n)
for b in np.unique(ofat_block):
    idx = np.where(ofat_block == b)[0]
    vals = df[b].to_numpy()[idx]
    order = np.argsort(vals)
    m = len(idx)
    if m == 1:
        pos = np.array([0.5])
    else:
        ranks = np.empty(m)
        ranks[order] = np.arange(m)
        pos = ranks / (m - 1)
    extremity[idx] = np.abs(pos - 0.5) * 2

rng = np.random.default_rng(SEED)
jitter = rng.uniform(-1e-9, 1e-9, size=n)
order = np.argsort(-(extremity + jitter))

split = np.array(["train"] * n)
split[order[:6]] = "test"
split[order[6:12]] = "val"

df = df.with_columns(pl.Series("extremity", extremity), pl.Series("split", split))

from collections import Counter
split_counts = Counter(split)
pl.DataFrame({"split": list(split_counts.keys()), "n": list(split_counts.values())}) \
    .sort("n", descending=True)


**Visual check** — the score that actually drove the split, one point per row:

In [ ]:
row_ids = np.arange(n)
split_colors = {"train": "#B7C9DA", "val": "#854F0B", "test": "#993C1D"}

fig = go.Figure()
for s, color in split_colors.items():
    mask = split == s
    fig.add_trace(go.Scatter(x=row_ids[mask], y=extremity[mask], mode="markers",
                              marker=dict(color=color, size=10), name=s))
fig.add_hline(y=extremity[order[11]], line_dash="dot", line_color="#5A5A55",
              annotation_text="val/train cutoff")
fig.update_layout(title="Extremity score per row, coloured by resulting split",
                   xaxis_title="row index", yaxis_title="extremity score",
                   width=750, height=400)
fig.show()


## 4. Check split composition — sizes and OFAT-block coverage

In [ ]:
from collections import Counter

for s in ["train", "val", "test"]:
    sub = df.filter(pl.col("split") == s)
    counts = Counter(sub["ofat_block"].to_list())
    print(f"--- {s}  (n={sub.shape[0]}) ---")
    for block, cnt in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f"    {block:9s} {cnt}")
    print()


In [ ]:
print("Global-extreme coverage per input (which split holds the row with the")
print("dataset's overall min / max of that input, not just its block's):\n")
for c in INPUT_COLS:
    min_row = df.filter(pl.col(c) == pl.col(c).min())
    max_row = df.filter(pl.col(c) == pl.col(c).max())
    print(f"{c:9s} global min -> split={min_row['split'][0]:5s} | "
          f"global max -> split={max_row['split'][0]}")


## 5. Min-max normalization (Eq. 3.1) — fit on train only

$$
x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
$$

$x_{\min}$/$x_{\max}$ are computed **from the training rows only** and
then applied to all three splits, so no information about the
validation or test rows leaks into the scaling — the requirement
Sec. 3.1.2.3 already states in words.

In [ ]:
train = df.filter(pl.col("split") == "train")
train_min = {c: train[c].min() for c in ALL_COLS}
train_max = {c: train[c].max() for c in ALL_COLS}

norm_exprs = [
    ((pl.col(c) - train_min[c]) / (train_max[c] - train_min[c])).alias(f"{c}_norm")
    for c in ALL_COLS
]
df = df.with_columns(norm_exprs)
df.select(["split"] + [f"{c}_norm" for c in ALL_COLS]).head()


**Before vs. after**, raw units on the left and the same points normalized on the right, for every variable:

In [ ]:
fig = make_subplots(rows=3, cols=3, subplot_titles=[f"{c} (raw -> norm)" for c in ALL_COLS])
split_colors = {"train": "#185FA5", "val": "#854F0B", "test": "#993C1D"}

for i, c in enumerate(ALL_COLS):
    r, cc = divmod(i, 3)
    for s, color in split_colors.items():
        sub = df.filter(pl.col("split") == s)
        fig.add_trace(
            go.Scatter(x=sub[c].to_numpy(), y=sub[f"{c}_norm"].to_numpy(), mode="markers",
                       marker=dict(color=color, size=7), name=s, showlegend=(i == 0)),
            row=r + 1, col=cc + 1,
        )
    fig.add_hline(y=0, line_dash="dot", line_color="#B0AFA8", row=r + 1, col=cc + 1)
    fig.add_hline(y=1, line_dash="dot", line_color="#B0AFA8", row=r + 1, col=cc + 1)

fig.update_layout(height=850, width=950,
                   title_text="Raw value (x-axis) vs. train-normalized value (y-axis)")
fig.show()


## 6. Where val/test normalized values land outside [0, 1]

Because test/val intentionally hold the domain's boundary points,
normalizing by the (narrower) train-only range will legitimately push
some of those values below 0 or above 1. That's expected here, not a
bug — it's the model being asked to handle points slightly beyond what
it was fit on, which is exactly what a boundary-aware test set is for.
Worth flagging explicitly rather than discovering it silently later.

In [ ]:
rows = []
for c in ALL_COLS:
    col = f"{c}_norm"
    non_train = df.filter(pl.col("split") != "train")
    below = non_train.filter(pl.col(col) < 0).shape[0]
    above = non_train.filter(pl.col(col) > 1).shape[0]
    rows.append({"variable": c, "below_0": below, "above_1": above,
                 "train_min": round(train_min[c], 4), "train_max": round(train_max[c], 4)})
range_check = pl.DataFrame(rows)
range_check


**Same table, as a picture** — how many val/test points fall below 0 or above 1 after train-only scaling:

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(x=range_check["variable"], y=range_check["below_0"],
                      name="below 0", marker_color="#993C1D"))
fig.add_trace(go.Bar(x=range_check["variable"], y=range_check["above_1"],
                      name="above 1", marker_color="#854F0B"))
fig.update_layout(barmode="group", title="Val/test points outside [0,1] after train-only normalization",
                   yaxis_title="count", width=750, height=400)
fig.show()


## 7. Train vs. validation vs. test per variable

Smooth KDE curve for train (n=28 supports a reasonable density
estimate); validation and test are shown as rug ticks at their actual
values rather than their own KDE curves, since a density estimate from
only 6 points would be more noise than signal.

In [ ]:
from scipy.stats import gaussian_kde

fig = make_subplots(rows=3, cols=3, subplot_titles=ALL_COLS)
colors = {"train": "#185FA5", "val": "#854F0B", "test": "#993C1D"}

for i, col in enumerate(ALL_COLS):
    r, c = divmod(i, 3)
    train_vals = df.filter(pl.col("split") == "train")[col].to_numpy()
    val_vals = df.filter(pl.col("split") == "val")[col].to_numpy()
    test_vals = df.filter(pl.col("split") == "test")[col].to_numpy()

    kde = gaussian_kde(train_vals)
    x_grid = np.linspace(df[col].min(), df[col].max(), 200)
    density = kde(x_grid)
    peak = density.max()

    fig.add_trace(go.Scatter(x=x_grid, y=density, mode="lines",
                              line=dict(color=colors["train"], width=2),
                              name="train", showlegend=(i == 0)),
                  row=r + 1, col=c + 1)
    fig.add_trace(go.Scatter(x=val_vals, y=[-0.06 * peak] * len(val_vals), mode="markers",
                              marker=dict(color=colors["val"], symbol="line-ns", size=11,
                                          line=dict(width=2, color=colors["val"])),
                              name="val", showlegend=(i == 0)),
                  row=r + 1, col=c + 1)
    fig.add_trace(go.Scatter(x=test_vals, y=[-0.14 * peak] * len(test_vals), mode="markers",
                              marker=dict(color=colors["test"], symbol="line-ns", size=11,
                                          line=dict(width=2, color=colors["test"])),
                              name="test", showlegend=(i == 0)),
                  row=r + 1, col=c + 1)

fig.update_layout(height=850, width=950,
                   title_text="Train KDE with validation/test rug ticks, per variable")
fig.show()


## 8. Split assignment over the OFAT structure (extends A1's plot)

In [ ]:
row_ids = np.arange(n)
split_colors = {"train": "#B7C9DA", "val": "#854F0B", "test": "#993C1D"}
split_arr = df["split"].to_numpy()

fig = make_subplots(rows=2, cols=2, subplot_titles=INPUT_COLS)
for i, col in enumerate(INPUT_COLS):
    r, c = divmod(i, 2)
    col_vals = df[col].to_numpy()
    for s, color in split_colors.items():
        mask = split_arr == s
        fig.add_trace(
            go.Scatter(x=row_ids[mask], y=col_vals[mask], mode="markers",
                       marker=dict(color=color, size=9), name=s, showlegend=(i == 0)),
            row=r + 1, col=c + 1,
        )
fig.update_layout(height=650, width=800,
                   title_text="Inputs vs. row index, coloured by split assignment")
fig.show()


## Optional — persist outputs

Saves row-level split/normalization info and the train-only scaling
parameters (needed again in every later stage, and in Phase D to
un-normalize predictions back to physical units).

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
df.write_csv(OUT_DIR / "A3_split_normalized.csv")

scaling = pl.DataFrame({
    "variable": ALL_COLS,
    "train_min": [train_min[c] for c in ALL_COLS],
    "train_max": [train_max[c] for c in ALL_COLS],
})
scaling.write_csv(OUT_DIR / "A3_scaling_params.csv")
print(f"Saved to {OUT_DIR}")


## Next

**Phase B (B1 — architecture search)** and **Phase C (C1 — physics
constraints)** both branch from here: B1 needs `split` to build its
5-fold CV over the train+val rows, and C1's collocation sampling (LHS)
should stay inside the train-derived normalized range from Section 5
above, not the full dataset range.